# SHAP Feature Importance Evaluation
This notebook securely unpacks the exact `artifacts` used in production, maps the readable Hebrew names, isolates the official mathematical test scope, and calculates exact SHAP baseline numbers.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
from IPython.display import display

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_loader import RealExcelDataLoader
from src.config import set_seed, FEATURE_DESCRIPTIONS
set_seed(42)

### 1. Configure Target Run & Extract Pipeline

In [ ]:
DATASET_NAME = "first_file"  # Options: 'first_file', 'second_file'
SPLIT_TYPE = "preset"        # Options: 'preset', 'random'

pipeline_path = f"../artifacts/model_pipeline_{DATASET_NAME}_{SPLIT_TYPE}.pkl"
raw_data_path = f"../data/{DATASET_NAME}/{DATASET_NAME}.xlsx"
test_data_path = f"../data/{DATASET_NAME}/test_data.xlsx"

print(f"Loading pipeline from: {pipeline_path}")
pipeline = joblib.load(pipeline_path)

best_model = pipeline['model']
scaler = pipeline['scaler']
feature_names = pipeline['feature_names']
dataset_config = pipeline.get('dataset_config', {})

print(f"\nAlgorithm extracted: {type(best_model).__name__}")

### 2. Isolate Formal Test Data & Apply Scalers

In [ ]:
loader = RealExcelDataLoader(raw_data_path, **dataset_config)
loader.scaler = scaler
loader.feature_names = feature_names

raw_df = loader.load()
emp_col = dataset_config.get('employee_id_col', 'fictive2')
if emp_col not in raw_df.columns:
    emp_col = 'fictive-oved' if 'fictive-oved' in raw_df.columns else emp_col
dataset_config['employee_id_col'] = emp_col

processed_df = loader.preprocess(raw_df, is_inference=True)
X_all = processed_df[feature_names]

if os.path.exists(test_data_path):
    test_df = pd.read_excel(test_data_path)
    test_ids = set(test_df[emp_col].astype(float).astype(int).astype(str))
    kept_ids = [str(int(float(eid))) for eid in loader.get_kept_indices()]
    test_mask = [eid in test_ids for eid in kept_ids]
    X_shap = X_all[test_mask].copy()
else:
    X_shap = X_all.copy()

# Force complete numerical evaluation to prevent SHAP crashing on booleans or mixed-types
X_shap = X_shap.apply(pd.to_numeric, errors='coerce').fillna(0.0).astype(np.float64)

# Build mapping for readable graphics (Do NOT overwrite columns so internal algorithm doesn't break)
readable_columns = [FEATURE_DESCRIPTIONS.get(c, c) for c in X_shap.columns]

### 3. Generate Mathematical SHAP Numbers

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

raw_model = getattr(best_model, 'model', best_model)

print("Constructing internal Baseline Metrics...\n")

if isinstance(raw_model, VotingClassifier):
    print("Using heavy PermutationExplainer on VotingClassifier... Sub-sampling test iterations for speed!")
    # Sample the SHAP dataset down heavily so it finishes in 5 seconds instead of 5 minutes!
    if len(X_shap) > 150:
        sample_idx = X_shap.sample(150, random_state=42).index
        X_shap = X_shap.loc[sample_idx]
        readable_columns = [readable_columns[i] for i, col in enumerate(X_all.columns)]
    
    background = X_shap.sample(min(30, len(X_shap)), random_state=42)
    def proba_churn(X):
        return raw_model.predict_proba(X)[:, 1]
    explainer = shap.PermutationExplainer(proba_churn, background)
    
    # Max_evals limits how many feature permutations it bothers trying before mathematically guessing
    raw_out = explainer(X_shap, max_evals=500)
    shap_values = raw_out.values
elif XGBClassifier and isinstance(raw_model, XGBClassifier):
    import xgboost as xgb
    dmat = xgb.DMatrix(X_shap)
    shap_values = raw_model.get_booster().predict(dmat, pred_contribs=True)
    shap_values = shap_values[:, :-1]
elif isinstance(raw_model, (RandomForestClassifier, AdaBoostClassifier)):
    explainer = shap.TreeExplainer(raw_model)
    shap_values_raw = explainer.shap_values(X_shap)
    shap_values = shap_values_raw[1] if isinstance(shap_values_raw, list) else (shap_values_raw[:,:,1] if shap_values_raw.ndim == 3 else shap_values_raw)
else:
    background = X_shap.sample(min(100, len(X_shap)), random_state=42)
    explainer = shap.LinearExplainer(raw_model, background)
    shap_values_raw = explainer.shap_values(X_shap)
    shap_values = shap_values_raw[1] if isinstance(shap_values_raw, list) else (shap_values_raw[:,:,1] if shap_values_raw.ndim == 3 else shap_values_raw)

# Calculate and print exact mathematical rankings
mean_abs_shap = np.abs(shap_values).mean(axis=0)
results = pd.DataFrame({
    'Feature_Readable': readable_columns,
    'Internal_Variable': feature_names,
    'Mean_Abs_SHAP': mean_abs_shap,
})
results = results.sort_values('Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
results.index += 1
display(results.head(40))

### 4. Global Topology Graphics

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=readable_columns, plot_type="bar", max_display=15)

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=readable_columns, max_display=20)